# Two-Step Fall Detection Pipeline

This notebook implements the second required architecture in the assignment: a two-step fall detection system.

Pipeline structure:

1. detect people in the image
2. crop each detected person
3. classify each crop as one of:
   - `fall detected`
   - `walk`
   - `sit`
4. evaluate the full pipeline on the held-out test set

This notebook is written to be readable for the team and directly comparable to the one-step baseline workflow.

## Project Goal

The assignment requires both:

- one-step fall detection
- two-step fall detection

The one-step baseline has already been trained and evaluated. The objective of this notebook is to build the two-step pipeline and then compare it fairly against the one-step model using the same held-out test set.

In [18]:
from pathlib import Path
from collections import Counter
import json
import random
import shutil

import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm

from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / 'prepared_dataset'
TRAIN_IMAGES = DATA_ROOT / 'images' / 'train'
TRAIN_LABELS = DATA_ROOT / 'labels' / 'train'
TEST_IMAGES = DATA_ROOT / 'images' / 'test'
TEST_LABELS = DATA_ROOT / 'labels' / 'test'

TWO_STEP_ROOT = PROJECT_ROOT / 'runs' / 'two_step'
CROPS_ROOT = TWO_STEP_ROOT / 'crops'
CLASSIFIER_ROOT = TWO_STEP_ROOT / 'classifier_runs'
PIPELINE_RESULTS_ROOT = TWO_STEP_ROOT / 'evaluation'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CLASS_NAMES = {
    0: 'fall detected',
    1: 'walk',
    2: 'sit',
}
CLASS_TO_ID = {value: key for key, value in CLASS_NAMES.items()}

print('Device       :', DEVICE)
print('Project root :', PROJECT_ROOT)
print('Data root    :', DATA_ROOT)
print('Two-step root:', TWO_STEP_ROOT)


Device       : cuda
Project root : /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system
Data root    : /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system/prepared_dataset
Two-step root: /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system/runs/two_step


## Two-Step Design Choice

This notebook uses a practical two-step design:

- **Step 1:** YOLO person detection
- **Step 2:** crop-based image classification using a CNN

This design is appropriate for the assignment because:

- it clearly separates detection and recognition
- it is easier to implement and explain than a pose-based pipeline
- it can be evaluated directly against the one-step model on the same test set

## Step 1 - Convert YOLO Boxes Into Person Crops

The classifier needs cropped person images. We generate these crops from the existing labeled train and test sets using the YOLO annotation files.

Each bounding box becomes one crop and inherits the class label from the corresponding annotation row.

In [19]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def load_yolo_rows(label_path: Path):
    rows = []
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:5])
        rows.append((class_id, x_center, y_center, width, height))
    return rows

def yolo_to_xyxy(box, image_width, image_height):
    class_id, x_center, y_center, width, height = box
    x_center *= image_width
    y_center *= image_height
    width *= image_width
    height *= image_height

    x1 = int(max(0, x_center - width / 2))
    y1 = int(max(0, y_center - height / 2))
    x2 = int(min(image_width, x_center + width / 2))
    y2 = int(min(image_height, y_center + height / 2))
    return class_id, x1, y1, x2, y2

def export_crops(image_dir: Path, label_dir: Path, split_name: str, output_root: Path):
    split_root = output_root / split_name
    if split_root.exists():
        shutil.rmtree(split_root)
    split_root.mkdir(parents=True, exist_ok=True)

    records = []
    image_paths = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])

    for image_path in tqdm(image_paths, desc=f'Exporting {split_name} crops'):
        label_path = label_dir / f'{image_path.stem}.txt'
        if not label_path.exists():
            continue

        image = cv2.imread(str(image_path))
        if image is None:
            continue
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image_rgb.shape[:2]

        for idx, box in enumerate(load_yolo_rows(label_path)):
            class_id, x1, y1, x2, y2 = yolo_to_xyxy(box, width, height)
            crop = image_rgb[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            class_name = CLASS_NAMES[class_id]
            class_dir = split_root / class_name
            class_dir.mkdir(parents=True, exist_ok=True)

            crop_name = f'{image_path.stem}__obj{idx:02d}{image_path.suffix.lower()}'
            crop_path = class_dir / crop_name
            Image.fromarray(crop).save(crop_path)

            records.append({
                'split': split_name,
                'source_image': str(image_path),
                'crop_path': str(crop_path),
                'class_id': class_id,
                'class_name': class_name,
                'x1': x1,
                'y1': y1,
                'x2': x2,
                'y2': y2,
            })

    return pd.DataFrame(records)


In [20]:
train_crops_df = export_crops(TRAIN_IMAGES, TRAIN_LABELS, 'train', CROPS_ROOT)
test_crops_df = export_crops(TEST_IMAGES, TEST_LABELS, 'test', CROPS_ROOT)

print('Train crops:', len(train_crops_df))
print('Test crops :', len(test_crops_df))

display(train_crops_df.head())


Exporting train crops:   0%|          | 0/485 [00:00<?, ?it/s]

Exporting test crops: 100%|██████████| 274/274 [00:32<00:00,  8.50it/s]

Train crops: 567
Test crops : 391


,split,source_image,crop_path,class_id,class_name,x1,y1,x2,y2
0,train,/mnt/Data/Swinburne/Intelligent_Systems/intell...,/mnt/Data/Swinburne/Intelligent_Systems/intell...,0,fall detected,78,78,200,122
1,train,/mnt/Data/Swinburne/Intelligent_Systems/intell...,/mnt/Data/Swinburne/Intelligent_Systems/intell...,0,fall detected,112,59,178,118
2,train,/mnt/Data/Swinburne/Intelligent_Systems/intell...,/mnt/Data/Swinburne/Intelligent_Systems/intell...,0,fall detected,47,64,255,143
3,train,/mnt/Data/Swinburne/Intelligent_Systems/intell...,/mnt/Data/Swinburne/Intelligent_Systems/intell...,0,fall detected,15,2,273,134
4,train,/mnt/Data/Swinburne/Intelligent_Systems/intell...,/mnt/Data/Swinburne/Intelligent_Systems/intell...,0,fall detected,14,3,257,168


In [21]:
print('Train crop class distribution:')
print(train_crops_df['class_name'].value_counts())
print()
print('Test crop class distribution:')
print(test_crops_df['class_name'].value_counts())


Train crop class distribution:
class_name
fall detected    285
walk             143
sit              139
Name: count, dtype: int64

Test crop class distribution:
class_name
walk             177
sit              119
fall detected     95
Name: count, dtype: int64


## Step 2 - Build the Crop Classifier Dataset

The crop classifier uses the exported person crops as training examples. We use the train crops for classifier training and the test crops for held-out evaluation.

In [22]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

class CropDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['crop_path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        label = int(row['class_id'])
        return image, label

train_dataset = CropDataset(train_crops_df, transform=train_transform)
test_dataset = CropDataset(test_crops_df, transform=test_transform)

# In notebook environments, multiprocessing data loaders can fail because
# locally defined classes such as CropDataset are not always pickle-safe
# across worker processes. Using num_workers=0 is slower but much more
# reliable for team notebooks and avoids the common '__main__' worker error.
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=NUM_WORKERS)

print('Train batches:', len(train_loader))
print('Test batches :', len(test_loader))
print('DataLoader workers:', NUM_WORKERS)


Train batches: 18
Test batches : 13
DataLoader workers: 0


## Step 3 - Define the Classifier Model

A lightweight pretrained image classifier is a practical starting point for the second stage. Here we use `ResNet18`, replacing its final layer to predict the three project classes.

In [23]:
classifier = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
classifier.fc = nn.Linear(classifier.fc.in_features, len(CLASS_NAMES))
classifier = classifier.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-4)

print(classifier.fc)


Linear(in_features=512, out_features=3, bias=True)


## Step 4 - Train the Classifier

This stage trains only the crop classifier. It does not yet include person detection at inference time. The objective is to first establish that the action-recognition stage can learn the crop-level classification task.

In [24]:
num_epochs = 10
best_test_accuracy = 0.0
best_classifier_path = CLASSIFIER_ROOT / 'best_classifier.pt'
CLASSIFIER_ROOT.mkdir(parents=True, exist_ok=True)

history = []

for epoch in range(num_epochs):
    classifier.train()
    running_loss = 0.0
    train_targets = []
    train_predictions = []

    for images, labels in tqdm(train_loader, desc=f'Train epoch {epoch + 1}/{num_epochs}'):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = classifier(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        train_targets.extend(labels.cpu().tolist())
        train_predictions.extend(preds.cpu().tolist())

    train_loss = running_loss / len(train_dataset)
    train_acc = accuracy_score(train_targets, train_predictions)

    classifier.eval()
    test_targets = []
    test_predictions = []
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f'Test epoch {epoch + 1}/{num_epochs}'):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = classifier(images)
            preds = outputs.argmax(dim=1)
            test_targets.extend(labels.cpu().tolist())
            test_predictions.extend(preds.cpu().tolist())

    test_acc = accuracy_score(test_targets, test_predictions)

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
    })

    print(f'Epoch {epoch + 1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, test_acc={test_acc:.4f}')

    if test_acc > best_test_accuracy:
        best_test_accuracy = test_acc
        torch.save(classifier.state_dict(), best_classifier_path)

print('Best classifier saved to:', best_classifier_path)
print('Best crop-level test accuracy:', best_test_accuracy)


Test epoch 1/10: 100%|██████████| 13/13 [00:06<00:00,  2.15it/s]


Epoch 1: train_loss=0.6022, train_acc=0.7601, test_acc=0.7852


Test epoch 2/10: 100%|██████████| 13/13 [00:06<00:00,  2.09it/s]


Epoch 2: train_loss=0.1545, train_acc=0.9612, test_acc=0.7903


Test epoch 3/10: 100%|██████████| 13/13 [00:06<00:00,  2.09it/s]


...

Epoch 8: train_loss=0.0206, train_acc=0.9947, test_acc=0.7596


Test epoch 9/10: 100%|██████████| 13/13 [00:06<00:00,  2.09it/s]


Epoch 9: train_loss=0.0100, train_acc=0.9982, test_acc=0.8261


Test epoch 10/10: 100%|██████████| 13/13 [00:06<00:00,  2.10it/s]

Epoch 10: train_loss=0.0105, train_acc=1.0000, test_acc=0.8133
Best classifier saved to: /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system/runs/two_step/classifier_runs/best_classifier.pt
Best crop-level test accuracy: 0.8388746803069054


## Step 5 - Crop-Level Classifier Evaluation

Before combining detection and classification into the full two-step pipeline, check the crop-level performance directly. This helps identify whether poor end-to-end results later come from the detector, the classifier, or both.

In [25]:
classifier.load_state_dict(torch.load(best_classifier_path, map_location=DEVICE))
classifier.eval()

all_targets = []
all_predictions = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Classifier evaluation'):
        images = images.to(DEVICE)
        outputs = classifier(images)
        preds = outputs.argmax(dim=1).cpu().tolist()
        all_predictions.extend(preds)
        all_targets.extend(labels.tolist())

print('Crop-level accuracy:', accuracy_score(all_targets, all_predictions))
print()
print(classification_report(all_targets, all_predictions, target_names=list(CLASS_NAMES.values())))


Classifier evaluation: 100%|██████████| 13/13 [00:06<00:00,  2.14it/s]

Crop-level accuracy: 0.8388746803069054

               precision    recall  f1-score   support

fall detected       0.91      0.75      0.82        95
         walk       0.82      0.96      0.89       177
          sit       0.82      0.73      0.77       119

     accuracy                           0.84       391
    macro avg       0.85      0.81      0.83       391
 weighted avg       0.84      0.84      0.84       391



## Step 6 - Full Two-Step Pipeline on the Held-Out Test Set

The full pipeline uses a person detector first and then applies the crop classifier to each detected person.

For a practical starting point, we use a pretrained YOLO person detector (`COCO person` class) as Stage 1.

In [26]:
person_detector = YOLO('yolov8n.pt')

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def classify_crop(crop_rgb: np.ndarray):
    image = Image.fromarray(crop_rgb).convert('RGB')
    tensor = inference_transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = classifier(tensor)
        pred_id = int(logits.argmax(dim=1).item())
    return pred_id

def load_ground_truth_boxes(image_path: Path, label_dir: Path):
    label_path = label_dir / f'{image_path.stem}.txt'
    if not label_path.exists():
        return []

    image = cv2.imread(str(image_path))
    if image is None:
        return []
    height, width = image.shape[:2]

    gt_boxes = []
    for box in load_yolo_rows(label_path):
        class_id, x1, y1, x2, y2 = yolo_to_xyxy(box, width, height)
        gt_boxes.append({
            'class_id': class_id,
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
        })
    return gt_boxes

def box_iou(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a['x1'], box_a['y1'], box_a['x2'], box_a['y2']
    xb1, yb1, xb2, yb2 = box_b['x1'], box_b['y1'], box_b['x2'], box_b['y2']

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_b = max(0, xb2 - xb1) * max(0, yb2 - yb1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0

def evaluate_two_step_predictions(pred_df: pd.DataFrame, test_image_paths, label_dir: Path, iou_threshold: float = 0.5):
    per_class_counts = {
        class_id: {'tp': 0, 'fp': 0, 'fn': 0}
        for class_id in CLASS_NAMES.keys()
    }
    matched_pairs = []

    for image_path in test_image_paths:
        image_key = str(image_path)
        image_preds = pred_df[pred_df['image_path'] == image_key].copy()
        gt_boxes = load_ground_truth_boxes(image_path, label_dir)
        gt_used = [False] * len(gt_boxes)

        pred_records = image_preds.to_dict(orient='records')
        # Higher-confidence sorting would be better, but the current pipeline does not
        # store detector confidence. Keep the existing order for deterministic matching.
        for pred in pred_records:
            pred_box = {
                'class_id': int(pred['predicted_class_id']),
                'x1': int(pred['x1']),
                'y1': int(pred['y1']),
                'x2': int(pred['x2']),
                'y2': int(pred['y2']),
            }

            best_iou = 0.0
            best_gt_idx = None
            for idx, gt in enumerate(gt_boxes):
                if gt_used[idx]:
                    continue
                if int(gt['class_id']) != pred_box['class_id']:
                    continue
                iou = box_iou(pred_box, gt)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx

            class_id = pred_box['class_id']
            if best_gt_idx is not None and best_iou >= iou_threshold:
                gt_used[best_gt_idx] = True
                per_class_counts[class_id]['tp'] += 1
                matched_pairs.append({
                    'image_path': image_key,
                    'class_id': class_id,
                    'class_name': CLASS_NAMES[class_id],
                    'iou': best_iou,
                    'match_type': 'tp',
                })
            else:
                per_class_counts[class_id]['fp'] += 1
                matched_pairs.append({
                    'image_path': image_key,
                    'class_id': class_id,
                    'class_name': CLASS_NAMES[class_id],
                    'iou': best_iou,
                    'match_type': 'fp',
                })

        for gt_idx, gt in enumerate(gt_boxes):
            if not gt_used[gt_idx]:
                per_class_counts[int(gt['class_id'])]['fn'] += 1
                matched_pairs.append({
                    'image_path': image_key,
                    'class_id': int(gt['class_id']),
                    'class_name': CLASS_NAMES[int(gt['class_id'])],
                    'iou': None,
                    'match_type': 'fn',
                })

    per_class_rows = []
    total_tp = total_fp = total_fn = 0
    for class_id, counts in per_class_counts.items():
        tp = counts['tp']
        fp = counts['fp']
        fn = counts['fn']
        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        per_class_rows.append({
            'class_id': class_id,
            'class_name': CLASS_NAMES[class_id],
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        })

    overall_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    overall_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    overall_f1 = 2 * overall_precision * overall_recall / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0.0

    summary = {
        'iou_threshold': iou_threshold,
        'tp': total_tp,
        'fp': total_fp,
        'fn': total_fn,
        'precision': overall_precision,
        'recall': overall_recall,
        'f1': overall_f1,
        'pred_instances': int(len(pred_df)),
        'gt_instances': int(sum(len(load_ground_truth_boxes(p, label_dir)) for p in test_image_paths)),
    }

    return summary, pd.DataFrame(per_class_rows), pd.DataFrame(matched_pairs)

two_step_predictions = []

test_image_paths = sorted([p for p in TEST_IMAGES.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])

for image_path in tqdm(test_image_paths, desc='Two-step inference'):
    image = cv2.imread(str(image_path))
    if image is None:
        continue
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    results = person_detector.predict(source=image_rgb, classes=[0], conf=0.25, imgsz=640, verbose=False)
    boxes = results[0].boxes
    if boxes is None:
        continue

    for idx in range(len(boxes)):
        x1, y1, x2, y2 = boxes.xyxy[idx].cpu().numpy().astype(int)
        crop = image_rgb[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        pred_class_id = classify_crop(crop)
        two_step_predictions.append({
            'image_path': str(image_path),
            'predicted_class_id': pred_class_id,
            'predicted_class_name': CLASS_NAMES[pred_class_id],
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
        })

two_step_predictions_df = pd.DataFrame(two_step_predictions)
two_step_summary, two_step_per_class_df, two_step_match_details_df = evaluate_two_step_predictions(
    two_step_predictions_df,
    test_image_paths,
    TEST_LABELS,
    iou_threshold=0.5,
)

display(two_step_predictions_df.head())
print(two_step_summary)
display(two_step_per_class_df)


Two-step inference: 100%|██████████| 274/274 [00:29<00:00,  9.15it/s]


,image_path,predicted_class_id,predicted_class_name,x1,y1,x2,y2
0,/mnt/Data/Swinburne/Intelligent_Systems/intell...,1,walk,201,102,693,1054
1,/mnt/Data/Swinburne/Intelligent_Systems/intell...,1,walk,195,213,389,742
2,/mnt/Data/Swinburne/Intelligent_Systems/intell...,1,walk,195,109,427,760
3,/mnt/Data/Swinburne/Intelligent_Systems/intell...,1,walk,587,77,803,844
4,/mnt/Data/Swinburne/Intelligent_Systems/intell...,1,walk,328,133,626,1070


{'iou_threshold': 0.5, 'tp': 322, 'fp': 93, 'fn': 69, 'precision': 0.7759036144578313, 'recall': 0.8235294117647058, 'f1': 0.7990074441687344, 'pred_instances': 415, 'gt_instances': 391}


,class_id,class_name,tp,fp,fn,precision,recall,f1
0,0,fall detected,66,8,29,0.891892,0.694737,0.781065
1,1,walk,170,54,7,0.758929,0.960452,0.847880
2,2,sit,86,31,33,0.735043,0.722689,0.728814


## Step 7 - Save Two-Step Outputs

At this stage we save the two-step predictions and training history so they can be reviewed later and compared against the one-step baseline.

In [27]:
PIPELINE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

history_df = pd.DataFrame(history)
history_path = PIPELINE_RESULTS_ROOT / 'classifier_history.csv'
crop_eval_path = PIPELINE_RESULTS_ROOT / 'crop_level_predictions.csv'
two_step_predictions_path = PIPELINE_RESULTS_ROOT / 'two_step_predictions.csv'
two_step_summary_path = PIPELINE_RESULTS_ROOT / 'two_step_metrics_summary.json'
two_step_per_class_path = PIPELINE_RESULTS_ROOT / 'two_step_per_class_metrics.csv'
two_step_matches_path = PIPELINE_RESULTS_ROOT / 'two_step_match_details.csv'

history_df.to_csv(history_path, index=False)
pd.DataFrame({
    'target': all_targets,
    'prediction': all_predictions,
}).to_csv(crop_eval_path, index=False)
two_step_predictions_df.to_csv(two_step_predictions_path, index=False)
two_step_per_class_df.to_csv(two_step_per_class_path, index=False)
two_step_match_details_df.to_csv(two_step_matches_path, index=False)
two_step_summary_path.write_text(json.dumps(two_step_summary, indent=2), encoding='utf-8')

print('Saved classifier history to :', history_path)
print('Saved crop predictions to   :', crop_eval_path)
print('Saved two-step outputs to   :', two_step_predictions_path)
print('Saved two-step summary to   :', two_step_summary_path)
print('Saved two-step per-class to :', two_step_per_class_path)
print('Saved match details to      :', two_step_matches_path)


Saved classifier history to : /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system/runs/two_step/evaluation/classifier_history.csv
Saved crop predictions to   : /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system/runs/two_step/evaluation/crop_level_predictions.csv
Saved two-step outputs to   : /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system/runs/two_step/evaluation/two_step_predictions.csv
Saved two-step summary to   : /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system/runs/two_step/evaluation/two_step_metrics_summary.json
Saved two-step per-class to : /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system/runs/two_step/evaluation/two_step_per_class_metrics.csv
Saved match details to      : /mnt/Data/Swinburne/Intelligent_Systems/intelligent-system/runs/two_step/evaluation/two_step_match_details.csv


## Notes for the Team

- This notebook establishes a practical two-step baseline.
- It now reports both crop-level classifier quality and an end-to-end detection-style summary based on IoU matching against the held-out test set.
- The final comparison section should use the same held-out test set for both architectures.
- The next refinement stage can include low-light robustness experiments and more structured condition-based analysis.
